In [1]:
import numpy as np
import pandas as pd

| Table | Why we need it |
|---|---|
| `CaseMaster.csv` | The fact table — one row per FIR, with station, crime type, date, coordinates, gravity |
| `GravityOffence.csv` | Lookup to turn `GravityOffenceID` into `Heinous` / `Serious` / etc., so we can compute `heinous_count` |
| `ChargesheetDetails.csv` | Tells us how each case was finally resolved (`cstype`), which feeds `resolution_rate` |

Everything else in the 26 files (Employee, Unit, Rank, District...) is
relevant to *other* modules or to enrichment, but not to this aggregation.

In [2]:
cm = pd.read_csv("../synthetic_data/CaseMaster.csv")
gravity = pd.read_csv("../synthetic_data/GravityOffence.csv")
cs = pd.read_csv("../synthetic_data/ChargesheetDetails.csv")

In [3]:
print("CaseMaster shape:", cm.shape)
print("GravityOffence shape:", gravity.shape)
print("ChargesheetDetails shape:", cs.shape)

CaseMaster shape: (15000, 18)
GravityOffence shape: (4, 2)
ChargesheetDetails shape: (13228, 5)


### Confirming if:
- there are no nulls in the columns that are going to be grouped
- the ID's that are going to be aggregated aren't some wierd mix (for example if range offices are mixed with actual police stations)

In [5]:
print("Unique Stations: ",cm["PoliceStationID"].nunique())
print("Unique crime sub-heads: ",cm["CrimeMinorHeadID"].nunique())
print("Nulls in key columns: ")
print(cm[["PoliceStationID","CrimeMinorHeadID","GravityOffenceID","IncidentFromDate"]].isnull().sum())

Unique Stations:  129
Unique crime sub-heads:  63
Nulls in key columns: 
PoliceStationID     0
CrimeMinorHeadID    0
GravityOffenceID    0
IncidentFromDate    0
dtype: int64


Deriving `period` from IncidentFromDate (when crime actually happened). The period will be monthly because it gives a denser and more learnable signal, at the cose of shorter forecast horizons.

In [ ]:
cm.info()
cm.to_csv

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CaseMasterID         15000 non-null  int64  
 1   CrimeNo              15000 non-null  int64  
 2   CaseNo               15000 non-null  int64  
 3   CrimeRegisteredDate  15000 non-null  str    
 4   PolicePersonID       15000 non-null  int64  
 5   PoliceStationID      15000 non-null  int64  
 6   CaseCategoryID       15000 non-null  int64  
 7   GravityOffenceID     15000 non-null  int64  
 8   CrimeMajorHeadID     15000 non-null  int64  
 9   CrimeMinorHeadID     15000 non-null  int64  
 10  CaseStatusID         15000 non-null  int64  
 11  CourtID              15000 non-null  int64  
 12  IncidentFromDate     15000 non-null  str    
 13  IncidentToDate       15000 non-null  str    
 14  InfoReceivedPSDate   15000 non-null  str    
 15  latitude             15000 non-null  float64
 1

In [6]:
cm["period"] = cm["IncidentFromDate"].values.astype("datetime64[M]")
cm[['IncidentFromDate','period']].head()

ValueError: Unexpected value for 'dtype': 'datetime64[M]'. Must be 'datetime64[s]', 'datetime64[ms]', 'datetime64[us]', 'datetime64[ns]' or DatetimeTZDtype'.